# Inverse Modeling of Soil Volumetric Liquid Water Content (VWC)

This script performs inverse modeling by comparing simulated soil VWC from an ensemble of SVS model runs with observed field measurements. The goal is to assess the performance of different parameter sets in reproducing soil moisture dynamics at two depths (75mm and 1850mm).

## Methodology

1. **Data Loading:**
   - Loads simulated VWC data (75mm and 1850mm) from Parquet files generated by the SVS ensemble run.
   - Loads observed VWC data for two lysimeters (L1 and L2) at each depth from a Parquet file.
   - Filters the observed data to the relevant time period (April 1, 2019, to September 1, 2019).
   - Resamples the observed data to hourly intervals.

2. **Performance Evaluation:**
   - For each depth and lysimeter combination:
     - Filters out rows with missing observed data.
     - Calculates the coefficient of determination (R²) and root mean square error (RMSE) between simulated and observed VWC for each member of the ensemble, using parallel processing for efficiency.
     - Saves the results (`member`, `R²`, `RMSE`) to separate text files.

## File Paths

* **Simulated VWC:**
    - 75mm: `./Results/WSOIL75mm_5Params_Mar8.parquet`
    - 1850mm: `./Results/WSOIL1850mm_5Params_Mar8.parquet`
* **Observed VWC:**
    - `../Data/FieldSoilMoisture_4Enclosures_4Lysimeters_201806_202210.parquet`
* **Output Files (R² and RMSE):**
    - L1 75mm: `./Results/InverseModeling_L1_75mm.txt`
    - L2 75mm: `./Results/InverseModeling_L2_75mm.txt`
    - L1 1850mm: `./Results/InverseModeling_L1_1850mm.txt`
    - L2 1850mm: `./Results/InverseModeling_L2_1850mm.txt`

## Dependencies

- **Python libraries:** `pathlib`, `pandas`, `numpy`
- **Helper function:** `calc_r2_rmse_parallel` (assumed to be defined in a separate `helpers` module)

## Key Assumptions

- The observed and simulated VWC data have consistent units (e.g., volumetric water content).
- The helper function `calc_r2_rmse_parallel` is implemented correctly and handles parallel processing efficiently.

## Author

Alireza Amani

## Date

2024-07-22


In [2]:
# 1) Importing Libraries -------------------------------------------------------
from pathlib import Path
import pandas as pd
from helpers import calc_r2_rmse_parallel
# ______________________________________________________________________________
# 2) Variables -----------------------------------------------------------------

## 2.1) Paths
paths = dict(
    sim_wsoil_75mm = Path("./Results/WSOIL75mm_5Params_Mar8.parquet"),
    sim_wsoil_1850mm = Path("./Results/WSOIL1850mm_5Params_Mar8.parquet"),
    obs_wsoil = Path("../Data/FieldSoilMoisture_4Enclosures_4Lysimeters_201806_202210.parquet"),

    # text files to store the results
    inv_l1_75mm = Path("./Results/InverseModeling_L1_75mm.txt"),
    inv_l2_75mm = Path("./Results/InverseModeling_L2_75mm.txt"),
    inv_l1_1850mm = Path("./Results/InverseModeling_L1_1850mm.txt"),
    inv_l2_1850mm = Path("./Results/InverseModeling_L2_1850mm.txt"),
)

## t0 and t1 for inverse modeling
T0 = pd.Timestamp("2019-04-01 00:00:00+00:00")
T1 = pd.Timestamp("2019-09-01 00:00:00+00:00")

## number of cores
N_CORES = 8

### assertions
for k, v in paths.items():
    # skip if .txt
    if v.suffix == ".txt":
        continue
    assert v.exists(), f"Path does not exist: {v}"

# ______________________________________________________________________________
# 3) Load Data -----------------------------------------------------------------

## 3.1) Load Simulated Data
sim_wsoil_75mm = pd.read_parquet(paths["sim_wsoil_75mm"])
sim_wsoil_1850mm = pd.read_parquet(paths["sim_wsoil_1850mm"])

## 3.2) Load Observed Data
obs_wsoil = pd.read_parquet(paths["obs_wsoil"]).set_index("Date Time (UTC)")

### 75 mm L1 and L2
obs_wsoil_75mm_L1 = obs_wsoil.query("Unique_label_ == 'L1_75mm_ext_Moisture_percent'").copy()
obs_wsoil_75mm_L2 = obs_wsoil.query("Unique_label_ == 'L2_75mm_ext_Moisture_percent'").copy()

### 1850 mm L1 and L2
obs_wsoil_1850mm_L1 = obs_wsoil.query("Unique_label_ == 'L1_1850mm_int_Moisture_percent'").copy()
obs_wsoil_1850mm_L2 = obs_wsoil.query("Unique_label_ == 'L2_1850mm_int_Moisture_percent'").copy()

### only t0 and t1
obs_wsoil_75mm_L1 = obs_wsoil_75mm_L1.loc[T0:T1]
obs_wsoil_75mm_L2 = obs_wsoil_75mm_L2.loc[T0:T1]
obs_wsoil_1850mm_L1 = obs_wsoil_1850mm_L1.loc[T0:T1]
obs_wsoil_1850mm_L2 = obs_wsoil_1850mm_L2.loc[T0:T1]

### agg to hourly
obs_wsoil_75mm_L1 = obs_wsoil_75mm_L1.resample("h").mean(numeric_only=True)
obs_wsoil_75mm_L2 = obs_wsoil_75mm_L2.resample("h").mean(numeric_only=True)
obs_wsoil_1850mm_L1 = obs_wsoil_1850mm_L1.resample("h").mean(numeric_only=True)
obs_wsoil_1850mm_L2 = obs_wsoil_1850mm_L2.resample("h").mean(numeric_only=True)

### remove obs_wsoil
del obs_wsoil



# ______________________________________________________________________________
# 4) Main ----------------------------------------------------------------------
# L1 75mm
obs_data = obs_wsoil_75mm_L1.dropna()
with open(paths["inv_l1_75mm"], "w", encoding="utf-8") as f:
        f.write("member,r2,rmse\n")
calc_r2_rmse_parallel(
    sim_data=sim_wsoil_75mm,
    obs_data=obs_data,
    sim_col="WSOIL_3_WSOIL_4",
    obs_col="Reading",
    member_col="member",
    text_file=paths["inv_l1_75mm"],
    num_processes=N_CORES
)
print("Inverse Modeling L1 75mm done")


## L2 75mm
obs_data = obs_wsoil_75mm_L2.dropna()
with open(paths["inv_l2_75mm"], "w", encoding="utf-8") as f:
        f.write("member,r2,rmse\n")
calc_r2_rmse_parallel(
    sim_data=sim_wsoil_75mm,
    obs_data=obs_data,
    sim_col="WSOIL_3_WSOIL_4",
    obs_col="Reading",
    member_col="member",
    text_file=paths["inv_l2_75mm"],
    num_processes=N_CORES
)
print("Inverse Modeling L2 75mm done")

## L1 1850mm
obs_data = obs_wsoil_1850mm_L1.dropna()
with open(paths["inv_l1_1850mm"], "w", encoding="utf-8") as f:
        f.write("member,r2,rmse\n")
calc_r2_rmse_parallel(
    sim_data=sim_wsoil_1850mm,
    obs_data=obs_data,
    sim_col="WSOIL_42_WSOIL_43",
    obs_col="Reading",
    member_col="member",
    text_file=paths["inv_l1_1850mm"],
    num_processes=N_CORES
)
print("Inverse Modeling L1 1850mm done")

## L2 1850mm
obs_data = obs_wsoil_1850mm_L2.dropna()
with open(paths["inv_l2_1850mm"], "w", encoding="utf-8") as f:
        f.write("member,r2,rmse\n")
calc_r2_rmse_parallel(
    sim_data=sim_wsoil_1850mm,
    obs_data=obs_data,
    sim_col="WSOIL_42_WSOIL_43",
    obs_col="Reading",
    member_col="member",
    text_file=paths["inv_l2_1850mm"],
    num_processes=N_CORES
)
print("Inverse Modeling L2 1850mm done")
# ______________________________________________________________________________

Inverse Modeling L2 75mm done


Exception: Stop here